# Fine-tune DistilBERT on Colab (fallback for the transformer stage)

Use this if training on your own machine is too slow. It runs the same `src/train_transformer.py` on a Colab GPU.

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells in order. When asked, upload these 5 files from the project:
   - `src/common.py`, `src/train_transformer.py`
   - `data/splits/train.parquet`, `data/splits/val.parquet`, `data/splits/test.parquet`
3. Unzip the downloaded `bert_outputs.zip` into the project's `outputs/` folder, then run `python src/cascade.py` locally.

Uploading your own split files (instead of re-running `prepare_data.py` here) guarantees the predictions line up row-for-row with your local splits.

In [ ]:
!nvidia-smi -L
!pip install -q "transformers>=5" datasets scikit-learn pyarrow

In [ ]:
import os
from google.colab import files

os.makedirs("src", exist_ok=True)
os.makedirs("data/splits", exist_ok=True)

uploaded = files.upload()  # select the 5 files listed above
for name, data in uploaded.items():
    folder = "src" if name.endswith(".py") else "data/splits"
    with open(os.path.join(folder, name), "wb") as f:
        f.write(data)

print("src:", sorted(os.listdir("src")))
print("data/splits:", sorted(os.listdir("data/splits")))

In [ ]:
# fp16 mixed precision is switched on automatically on CUDA GPUs
!python src/train_transformer.py --epochs 2 --batch-size 32

In [ ]:
!cd outputs && zip -q ../bert_outputs.zip bert_val_proba.npy bert_test_proba.npy bert_metrics.json
files.download("bert_outputs.zip")